Notebook to join liu greenness to (GLAKES) and compute LEV measures

Prerequisite: fix GLAKES geoms, clip to desired domain

"""Originally did this with QGIS"""

In [1]:
import pandas as pd
from pathlib import Path
import geopandas as gpd
import seaborn as sns
from matplotlib import pyplot as plt
import os
import numpy as np
from shapely.geometry import box

from land_cover.load import GLAKES_filtered_fix_aqveg_dir

%load_ext autoreload
%autoreload 2

In [2]:
pth_glakes_ts = "/Volumes/metis/Datasets/GLAKES/GLAKES area time series.csv"

In [ ]:
df_csv_MA = pd.read_csv("/Volumes/metis/Datasets/Liu_aq_veg/figshare/MA.csv")
df_csv_NDVI = pd.read_csv("/Volumes/metis/Datasets/Liu_aq_veg/figshare/NDVI.csv")
df_csv_MA = df_csv_MA.merge(
    df_csv_NDVI[["Lake_id", "NDVI8499", "NDVI0010", "NDVI1121"]], on=["Lake_id"], how="outer"
)

# GLAKES spatial attributes in NA and Scandinavia
gdf_glakes = gpd.read_file("/Volumes/metis/Datasets/GLAKES/out/GLAKES_filtered_fix.shp")

# join in df_csv_MA/NDVI to get greenness
gdf = gdf_glakes.merge(
    df_csv_MA.drop(columns=["Lat", "Lon"]),
    on="Lake_id",
    how="left", # keep all lakes in study area, even though some may not have greenness for some reason (area = 0?)
)
# Join in time series, keep _wm, not _nm
gdf_ts = pd.read_csv(pth_glakes_ts)
gdf = gdf.merge(
    gdf_ts.drop(columns=["area_1984_1999_nm","area_2000_2009_nm","area_2010_2019_nm"]),
    on="Lake_id",
    how="left"
)

Note: some lakes even in N domain are missing aquatic veg estimates for some reason

In [11]:
df_csv_MA.query("Lake_id == 1005155")
gdf_glakes.query("Lake_id == 1005155")

,OBJECTID,Lake_id,Area_bound,Area_PW,Continent,Lat,Lon,GFed_flag,PFed_flag,Endo_flag,Rser_flag,Shape_Leng,Shape_Area,geometry
542082,1005155.0,1005155,0.166687,0.139891,North America,64.004207,-109.24159,0,1,0,0,0.03,0.000031,"POLYGON ((-109.23875 64.00725, -109.23875 64.0..."


In [12]:
len(df_csv_NDVI)

321989

In [13]:
len(df_csv_MA)

1110287

In [4]:
assert gdf["geometry"].notnull().all(), "Some geometry values in gdf are empty"
assert gdf_glakes["geometry"].notnull().all(), "Some geometry values in gdf_glakes are empty"

In [ ]:
# Add LEV stats
gdf["LEV_p1"] = gdf.areaP1 / gdf["area_1984_1999_wm"] * 100
gdf["LEV_p2"] = gdf.areaP2 / gdf["area_2000_2009_wm"] * 100
gdf["LEV_p3"] = gdf.areaP3 / gdf["area_2010_2019_wm"] * 100

for i in range(1, 4):
    gdf.loc[np.isinf(gdf[f"LEV_p{i}"]), f"LEV_p{i}"] = np.nan

gdf["LEV_p13ain"] = gdf.areaP3 - gdf.areaP1 # "LEV period 1 to period 3 absolute increase"
gdf["LEV_p13rin"] = gdf.LEV_p3 - gdf.LEV_p1 # "LEV period 1 to period 3 relative increase"
gdf["LEV_p13in"] = gdf["LEV_p13ain"] / gdf["area_1984_1999_wm"] * 100  # "Another type of relative increase, normalized to initial lake area

In [6]:
gdf.head()

,OBJECTID,Lake_id,Area_bound,Area_PW,Continent,Lat,Lon,GFed_flag,PFed_flag,Endo_flag,...,NDVI0010,NDVI1121,area_1984_1999_wm,area_2000_2009_wm,area_2010_2019_wm,LEV_p1,LEV_p2,LEV_p3,LEV_p13ain,LEV_p13rin
0,3.0,3,82155.420436,79514.496612,North America,47.526368,-87.757371,0,0,0,...,0.66,0.68,80903.091562,81679.557074,81192.035002,0.008512,0.007922,0.007740,-0.601649,-0.000771
1,8.0,8,30657.104741,28864.201812,North America,65.998658,-120.968462,0,1,0,...,0.55,0.58,30156.030260,30565.860504,30502.390581,0.001103,0.002080,0.003208,0.645770,0.002105
2,10.0,10,27480.339361,26652.927111,North America,52.825958,-97.738194,0,1,0,...,NaN,NaN,26977.563484,27182.922134,27256.549512,NaN,NaN,NaN,NaN,NaN
3,11.0,11,26827.549399,25926.211686,North America,61.769419,-113.811408,0,1,0,...,0.68,0.70,26559.357871,26710.565325,26650.593451,0.048299,0.055355,0.090343,11.249214,0.042045
4,16.0,16,17460.818811,16808.357727,Europe,60.829856,31.477989,0,0,0,...,0.70,0.72,17171.066362,17284.003108,17304.408409,0.113397,0.086365,0.090666,-3.782168,-0.022730


In [7]:
gdf.columns

Index(['OBJECTID', 'Lake_id', 'Area_bound', 'Area_PW', 'Continent', 'Lat',
       'Lon', 'GFed_flag', 'PFed_flag', 'Endo_flag', 'Rser_flag', 'Shape_Leng',
       'Shape_Area', 'geometry', 'areaP1', 'areaP2', 'areaP3', 'NDVI8499',
       'NDVI0010', 'NDVI1121', 'area_1984_1999_wm', 'area_2000_2009_wm',
       'area_2010_2019_wm', 'LEV_p1', 'LEV_p2', 'LEV_p3', 'LEV_p13ain',
       'LEV_p13rin'],
      dtype='object')

In [8]:
# write out
gdf.to_file(GLAKES_filtered_fix_aqveg_dir)

/var/folders/rv/sn0kln2103b9fs4xl56n3w7w0000gn/T/ipykernel_35807/2638634862.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(GLAKES_filtered_fix_aqveg_dir)
/Users/ekyzivat/mambaforge/envs/landcover/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'area_1984_1999_wm' to 'area_1984_'
  ogr_write(
/Users/ekyzivat/mambaforge/envs/landcover/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'area_2000_2009_wm' to 'area_2000_'
  ogr_write(
/Users/ekyzivat/mambaforge/envs/landcover/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Normalized/laundered field name: 'area_2010_2019_wm' to 'area_2010_'
  ogr_write(
/Users/ekyzivat/mambaforge/envs/landcover/lib/python3.11/site-packages/pyogrio/raw.py:723: RuntimeWarning: Value -56148430.0239839852 of field LEV_p13rin of feature 45588 not successfully written. Possibly 